# <span style="font-size: 20pt; font-weight: bold; color: #0098cd;">Procesamiento del lenguaje natural: clasificación y estructuración de datos</span>


## 1. Introducción

### 1.1 Objetivo
transformar texto en datos estructurados útiles para análisis y modelado

👉 siguiente paso: definir **estructura exacta del JSON final + diccionario de normalización LATAM**
Ahí es donde tu TFM pasa de bueno a muy sólido.

### 1.2 Contexto dentro del sistema completo

### 1.3 Requisitos de esta fase

  * clasificación de intención
  * extracción de entidades (NER)
  * normalización

## 2. Preparación del entorno de trabajo

### 2.1 Instalación de dependencias

En este apartado se especifica la instalación de las librerías necesarias para la ejecución del *notebook*. Todas las dependencias se gestionan mediante un archivo `requirements.txt`, lo que permite replicar el entorno de ejecución de forma controlada y consistente.

In [ ]:
# ============================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================
# Este notebook requiere la instalación previa de las dependencias del proyecto.
# Ejecutar en terminal:
# pip install -r requirements.txt

### 2.2 Importación de librerías

Se importan las librerías necesarias para las distintas fases del procesamiento, incluyendo manipulación de señal, tratamiento de audio, modelos de detección de voz, análisis de datos y visualización.

Se organiza por bloques funcionales para permitir identificar claramente el propósito de cada conjunto de herramientas dentro del *pipeline*, facilitando la mantenibilidad y comprensión del código.

In [ ]:
# ==============================
# Gestión de rutas y sistema
# ==============================
from pathlib import Path                  # gestión de rutas del sistema de archivos
import json                               # lectura y escritura de ficheros JSON

# ==============================
# Manipulación de datos
# ==============================
import pandas as pd                       # manejo de datos tabulares
import numpy as np                        # operaciones numéricas

# ==============================
# Procesamiento de texto
# ==============================
import re                                 # expresiones regulares
import unicodedata                        # normalización unicode
from unidecode import unidecode           # eliminación de acentos y normalización simplificada

# ==============================
# NLP - modelos y pipelines
# ==============================
import spacy                              # procesamiento lingüístico y NER
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # modelos tipo BERT (BETO)
from sklearn.model_selection import train_test_split   # partición de datos
from sklearn.linear_model import LogisticRegression    # modelo baseline de clasificación

# ==============================
# Normalización y matching
# ==============================
from rapidfuzz import fuzz, process       # comparación difusa de texto (matching de términos)

# ==============================
# Utilidades y control
# ==============================
from tqdm import tqdm                     # barra de progreso
import joblib                             # guardado y carga de modelos

### 2.3 Gestión y configuración de rutas

In [16]:
# Detectar raíz
project_root = Path.cwd()

while not (project_root / "data").exists():
    project_root = project_root.parent


# =========================
# DIRECTORIOS
# =========================

# Ruta raíz del proyecto
data_dir = project_root / "data"


# =========================
# TRANSCRIPCIONES (INPUT NLP)
# =========================

# Carpeta general de transcripciones
transcriptions_dir = data_dir / "transcriptions"

# Salida final del ASR (pipeline óptimo)
pred_asr_dir = transcriptions_dir / "asr_output"

# Ground truth (para evaluación)
ground_truth_dir = transcriptions_dir / "ground_truth"


# =========================
# SALIDA NLP (NUEVO)
# =========================

# Carpeta para datos estructurados generados
structured_data_dir = data_dir / "structured_data"

# Subcarpetas
classification_dir = structured_data_dir / "classification"
ner_dir = structured_data_dir / "ner"
final_output_dir = structured_data_dir / "final_output"


# =========================
# CREACIÓN DE CARPETAS
# =========================

classification_dir.mkdir(parents=True, exist_ok=True)
ner_dir.mkdir(parents=True, exist_ok=True)
final_output_dir.mkdir(parents=True, exist_ok=True)


# =========================
# VERIFICACIÓN
# =========================

print("Ruta raíz:", project_root)

print("\n--- Input NLP ---")
print("Transcripciones procesadas:", pred_processed_dir)
print("Ground truth:", ground_truth_dir)

print("\n--- Output NLP ---")
print("Clasificación:", classification_dir)
print("NER:", ner_dir)
print("Salida final:", final_output_dir)

Ruta raíz: /Volumes/EXTENSION/GitHub/TFM

--- Input NLP ---
Transcripciones procesadas: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/predictions/processed
Ground truth: /Volumes/EXTENSION/GitHub/TFM/data/transcriptions/ground_truth

--- Output NLP ---
Clasificación: /Volumes/EXTENSION/GitHub/TFM/data/structured_data/classification
NER: /Volumes/EXTENSION/GitHub/TFM/data/structured_data/ner
Salida final: /Volumes/EXTENSION/GitHub/TFM/data/structured_data/final_output


## 3. Preparación del dataset para NLP

### 3.1 Carga de transcripciones

In [26]:
# -------- ASR --------

# Carga de archivos JSON del ASR (pipeline final)
json_paths = sorted(list(pred_asr_dir.glob("*.json")))

dataset_asr = []

for path in json_paths:
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        dataset_asr.append({
            "audio_id": data.get("audio_id", path.stem),
            "text": data.get("text", "")
        })
        
    except Exception as e:
        dataset_asr.append({
            "audio_id": path.stem,
            "text": None,
            "error": str(e)
        })

# Conversión a DataFrame
df_asr = pd.DataFrame(dataset_asr)

print(f"ASR cargado: {len(df_asr)} registros")
df_asr.head()


# -------- GROUND TRUTH --------

# Carga de anotaciones manuales
gt_path = ground_truth_dir / "ground_truth.csv"

df_gt = pd.read_csv(gt_path)

print(f"Ground truth cargado: {len(df_gt)} registros")

# Estandarización de columnas para NLP
df_gt = df_gt.rename(columns={
    "transcripcion": "text",
    "clase": "label"
})

df_gt.head()

ASR cargado: 11 registros
Ground truth cargado: 11 registros


,audio_id,text,label
0,AUDIO-2026-04-22-11-24-06,Aquí probando un minuto de audio para ver si p...,registro_inicial
1,AUDIO-2026-04-25-10-35-47,"Bueno, por lo de la vacuna no me preocupa porq...",registro_inicial
2,AUDIO-2026-04-25-10-36-13,"Bueno es que estaba comiendo, estoy comiendo. ...",registro_inicial
3,AUDIO-2026-04-25-10-36-43,"Hola, hola, buen día. vean que estoy regresand...",registro_inicial
4,AUDIO-2026-04-22-11-23-01,Esto es una prueba de audio a ver qué tal se c...,registro_inicial


### 3.2 Validación de datos

#### 3.2.1 Validación del dataset ASR

In [27]:
# Texto válido: no vacío y tipo string
df_asr["text_valid"] = df_asr["text"].apply(
    lambda x: isinstance(x, str) and len(x.strip()) > 0
)

# Filtrado
valid_asr = df_asr[df_asr["text_valid"]].copy()
invalid_asr = df_asr[~df_asr["text_valid"]].copy()

print(f"ASR → válidos: {len(valid_asr)}, inválidos: {len(invalid_asr)}")

# Mostrar ejemplos inválidos si existen
if len(invalid_asr) > 0:
    print("\nEjemplos inválidos ASR:")
    display(invalid_asr.head())

ASR → válidos: 11, inválidos: 0


#### 3.2.2 Validación del dataset Ground Truth

In [29]:
# Texto válido
df_gt["text_valid"] = df_gt["text"].apply(
    lambda x: isinstance(x, str) and len(x.strip()) > 0
)

# Label válido (no nulo)
df_gt["label_valid"] = df_gt["label"].notnull()

# Filtrado
valid_gt = df_gt[(df_gt["text_valid"]) & (df_gt["label_valid"])].copy()
invalid_gt = df_gt[~((df_gt["text_valid"]) & (df_gt["label_valid"]))].copy()

print(f"GT → válidos: {len(valid_gt)}, inválidos: {len(invalid_gt)}")

# Mostrar ejemplos inválidos si existen
if len(invalid_gt) > 0:
    print("\nEjemplos inválidos GT:")
    display(invalid_gt.head())

GT → válidos: 11, inválidos: 0


### 3.3 Análisis inicial del dataset Ground Truth

In [30]:
print("Distribución de clases:")
print(valid_gt["label"].value_counts())

print("\nDistribución relativa:")
print(valid_gt["label"].value_counts(normalize=True))

Distribución de clases:
label
registro_inicial    11
Name: count, dtype: int64

Distribución relativa:
label
registro_inicial    1.0
Name: proportion, dtype: float64


## 4. Clasificación de mensajes (intención)

### 4.1 Definición de clases

* `registro_inicial`
* `evento_cultivo`
* `otro`

### 4.2 Preparación del dataset

* textos etiquetados por investigadores

### 4.3 Modelo

* fine-tuning ligero (BETO o similar)

### 4.4 Inferencia

* asignación de clase a cada texto

### 4.5 Evaluación

* métricas: exactitud, F1
* análisis de errores

## 5. Extracción de entidades (NER)

### 5.1 Definición de entidades

* `cultivo`, `producto`, `accion`, `problema`, `cantidad`, `fecha_evento`, etc.

### 5.2 Preparación del dataset

* anotación de textos

### 5.3 Enfoque

* modelo NER (spaCy / transformers)
* enfoque híbrido (modelo + reglas)

### 5.4 Inferencia

* extracción de entidades por texto

### 5.5 Evaluación

* precisión, recall, F1
* análisis de errores

## 6. Normalización de entidades (clave)

### 6.1 Problema

* variabilidad lingüística (LATAM)

### 6.2 Estrategia

* diccionario de sinónimos
* reglas de normalización

Ejemplo:

```text
abono / fertilizante → fertilizante
```

### 6.3 Aplicación

* transformación de entidades a formato estándar

## 7. Integración del pipeline

```text
texto → clasificación → NER → normalización → datos estructurados
```

* uso de la clase para guiar extracción

## 8. Generación de output final

```json
{
  "audio_id": "...",
  "user_id": "...",
  "transcription": "...",
  "clasificacion": "evento_cultivo",
  "entidades": {
    "accion": "...",
    "producto": "...",
    "cantidad": "...",
    "problema": "..."
  },
  "processing_version": "v1"
}
```

## 9. Análisis global de resultados

* distribución de clases
* calidad de entidades
* errores frecuentes

## 10. Selección del pipeline final

* justificación de:

  * modelo clasificación
  * modelo NER
  * reglas de normalización

## 11. Conclusiones
- Resumen del proceso aplicado
- Resultados principales
- Relevancia para el pipeline de Speech-to-Text

Te las defino, pero ojo: si no las acotas bien ahora, luego tendrás ruido y dataset inútil.

Voy a darte un **esquema mínimo pero sólido**, orientado a tu objetivo (modelo posterior).

---

# **Definición de entidades (NER) — Notebook 3**

## 🎯 Objetivo

Extraer información estructurada de los audios para construir un dataset útil para análisis y modelos predictivos.

---

## ✅ 1. Entidades de contexto (registro inicial)

Estas definen la “foto base” del productor:

* `cultivo` → café, cacao, maíz…
* `variedad` → arábica, robusta… (si aparece)
* `ubicacion` → finca, zona, comunidad
* `superficie` → hectáreas / extensión
* `edad_cultivo` → años / meses
* `sistema_produccion` → orgánico, convencional (si aplica)

---

## ✅ 2. Entidades de evento (núcleo del sistema)

Aquí está el valor real.

### 🔹 Acción

* `accion` → fertilización, riego, poda, fumigación, siembra, cosecha

### 🔹 Producto / insumo

* `producto` → urea, fungicida, abono, herbicida

### 🔹 Problema

* `problema` → plaga, enfermedad, sequía, exceso de lluvia

### 🔹 Cantidad

* `cantidad` → “2 sacos”, “5 litros”

### 🔹 Fecha / tiempo

* `fecha_evento` → ayer, hoy, hace 3 días

### 🔹 Estado del cultivo

* `estado` → bueno, malo, afectado, creciendo

---

## ✅ 3. Entidades ambientales (muy importantes)

* `clima` → lluvia, calor, sequía
* `condicion_suelo` → húmedo, seco (si aparece)

---

## ✅ 4. Entidades auxiliares (opcionales pero útiles)

* `frecuencia` → cada semana, una vez al mes
* `intensidad` → mucho, poco
* `observacion` → texto libre relevante

---

# 🧠 Ejemplo real

Audio:

> “ayer eché dos sacos de urea porque vi una plaga”

Salida:

```json
{
  "accion": "fertilizacion",
  "producto": "urea",
  "cantidad": "2 sacos",
  "fecha_evento": "ayer",
  "problema": "plaga"
}
```

